# Combinatorial CoWork 2026 — Notebook 02: Resolutions, `Hom`, `Ext`, and `Tor`

Audience:
- Participants who want the most commutative-algebra-flavored part of the library first: common encodings, resolutions, tables, and derived functors.

Learning goals:
- Build two tiny rectangle-indicator modules in `Z^2` through the flange interface.
- Common-encode them and compute `Hom`, `Ext`, `Tor`, projective resolutions, and injective resolutions.
- Use `betti_table(...)`, `bass_table(...)`, `degree_range(...)`, and `dim(...)` as the cheap-first inspection surfaces.


In [10]:
NOTEBOOK_STEM = "02_resolutions_hom_ext_tor"

_TO_ROOT = let
    dir = abspath(pwd())
    root = nothing
    while true
        if isfile(joinpath(dir, "src", "TamerOp.jl"))
            root = dir
            break
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate repo root containing src/TamerOp.jl from pwd()=$(pwd()).")
        dir = parent
    end
    root
end

if !isdefined(Main, :TO) || !isdefined(Main, :example_header)
    Base.include(Main, joinpath(_TO_ROOT, "examples", "_common.jl"))
end

CM = TO.CoreModules
OPT = TO.Options
sc = TO.SessionCache()
outdir = joinpath(_TO_ROOT, "examples", "_outputs", "combinatorial_cowork_2026", NOTEBOOK_STEM)
mkpath(outdir)



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

  ** incremental compilation may be broken for this module **


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
┌ Warning: FieldLinAlg: threshold fingerprint mismatch; using defaults. Run `FieldLinAlg.autotune_linalg_thresholds!()` to regenerate.
│   path = "/home/eriknovak/Documents/duke_fall_2025/tamer-op/linalg_thresholds.toml"
└ @ TamerOp.FieldLinAlg ~/Documents/duke_fall_2025/tamer-op/src/field_linalg/thresholds.jl:582

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
┌ Warning: FieldLinAlg: threshold fingerprint mismatch; using defaults. Run `FieldLinAlg.autotune_linalg_thresholds

"/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/02_resolutions_hom_ext_tor"

## 1. Build two rectangle-indicator flanges

This is the algebraic workhorse example for the talk. The rectangles are intentionally tiny so that exact arithmetic over `QQ` stays responsive. We use `SyntheticData.orthant_bar_flange(...)` directly here rather than a notebook-local helper.


In [11]:
FM = TO.SyntheticData.orthant_bar_flange(
    bars=[([0, 0], [2, 2])],
    field=CM.QQField(),
)

FN = TO.SyntheticData.orthant_bar_flange(
    bars=[([1, 1], [3, 3])],
    field=CM.QQField(),
)

enc_opts = OPT.EncodingOptions(backend=:zn, max_regions=50_000, poset_kind=:signature)

(; FM=TO.describe(FM), FN=TO.describe(FN), enc_opts=enc_opts)


(FM = (kind = :flange, ambient_dim = 2, field = "TamerOp.CoreModules.CoeffFields.QQField", nflats = 1, ninjectives = 1, matrix_size = (1, 1)), FN = (kind = :flange, ambient_dim = 2, field = "TamerOp.CoreModules.CoeffFields.QQField", nflats = 1, ninjectives = 1, matrix_size = (1, 1)), enc_opts = EncodingOptions(backend=:zn, poset_kind=:signature, max_regions=50000))

## 2. Common-encode the pair

`encode((FM, FN), enc_opts; ...)` is the notebook-facing path when two modules should be compared or combined on one common encoding. We also inspect one explicit geometry summary, not a hidden helper wrapper.


In [12]:
encM, encN = TO.encode((FM, FN), enc_opts; cache=sc)
regions_spec = TO.Visualization.visual_spec(TO.encoding_map(encM); kind=:regions)

(; M_summary=TO.describe(encM),
   N_summary=TO.describe(encN),
   shared_poset=(TO.encoding_poset(encM) === TO.encoding_poset(encN)),
   shared_classifier=(TO.encoding_map(encM) === TO.encoding_map(encN)),
   regions_visual_summary=TO.Visualization.visual_summary(regions_spec))


(M_summary = (kind = :encoding_result, poset_type = TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_map_type = TamerOp.EncodingCore.CompiledEncoding{TamerOp.ZnEncoding.ZnEncodingMap{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Int64}, Vector{Int64}}, Vector{Tuple{Int64, Int64}}, TamerOp.CoreModules.EncodingCache}, compiled = true, backend = :zn, has_cohomology = true, has_presentation = true, module_dims = [0, 0, 0, 0, 0, 1, 1, 0, 0]), N_summary = (kind = :encoding_result, poset_type = TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_map_type = TamerOp.EncodingCore.CompiledEncoding{TamerOp.ZnEncoding.ZnEncodingMap{2, 1, 1}, TamerOp.ZnEncodi

## 3. Projective and injective diagnostics

These are the cheapest standard commutative-algebra summaries to show before diving into full derived-functor objects.


In [13]:
res_opts = OPT.ResolutionOptions(maxlen=2, minimal=true, check=true)
res_proj = TO.resolve(encM; kind=:projective, opts=res_opts, cache=sc)
res_inj = TO.resolve(encN; kind=:injective, opts=res_opts, cache=sc)

(; projective=TO.result_summary(res_proj),
   injective=TO.result_summary(res_inj),
   betti=TO.betti_table(res_proj),
   bass=TO.bass_table(res_inj))


(projective = (kind = :resolution_result, resolution_type = TamerOp.DerivedFunctors.Resolutions.ProjectiveResolution{Rational{BigInt}}, source_type = TamerOp.Results.EncodingResult{TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, TamerOp.EncodingCore.CompiledEncoding{TamerOp.ZnEncoding.ZnEncodingMap{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Int64}, Vector{Int64}}, Vector{Tuple{Int64, Int64}}, TamerOp.CoreModules.EncodingCache}, TamerOp.FiniteFringe.FringeModule{Rational{BigInt}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}}, TamerOp.FlangeZn.Flange{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, 2}, @NamedTuple{}}, has_betti = true, has_minimality = false, opts_type = TamerOp.Options.ResolutionOptions), injective = (kind = :resolution_result, resolutio

## 4. `Hom`, `Ext`, and `Tor`

For the shared-encoding pair we first inspect `Hom` and `Ext`. `Tor` needs a right-module on the opposite poset, so we demonstrate it in a separate tiny explicit cell immediately after this one.


In [14]:
hom_dim = TO.hom_dimension(encM, encN; cache=sc)
H = TO.hom(encM, encN; cache=sc)
E = TO.ext(encM, encN; maxdeg=2, cache=sc)

(; hom_dim=hom_dim,
   hom_summary=TO.describe(H),
   ext_degrees=collect(TO.Advanced.degree_range(E)),
   ext_dims=[TO.Advanced.dim(E, t) for t in TO.Advanced.degree_range(E)])


(hom_dim = 0, hom_summary = (kind = :hom_space, field = TamerOp.CoreModules.CoeffFields.QQField(), nvertices = 9, dimension = 0, degree_range = 0:0, basis_cached = false, source_total_dim = 2, target_total_dim = 2), ext_degrees = [0, 1, 2], ext_dims = [0, 0, 0])

## 5. `Tor` needs an opposite-poset pair

To keep the contract honest, we build one tiny explicit right-module/left-module pair on opposite finite posets. This is small enough for a live notebook cell and makes the variance from `Hom`/`Ext` explicit.


In [15]:
TOA = TO.Advanced
P = TOA.FinitePoset(Bool[1 1; 0 1]; check=false)
Pop = TOA.FinitePoset(Bool[1 0; 1 1]; check=false)
idP = TOA.EncodingMap(P, P, [1, 2])
idPop = TOA.EncodingMap(Pop, Pop, [1, 2])
field = CM.QQField()
HL = TOA.one_by_one_fringe(P, TOA.principal_upset(P, 1), TOA.principal_downset(P, 2), TO.QQ(1); field=field)
HRop = TOA.one_by_one_fringe(Pop, TOA.principal_upset(Pop, 2), TOA.principal_downset(Pop, 1), TO.QQ(1); field=field)
encL = TO.EncodingResult(P, TOA.pmodule_from_fringe(HL), idP; H=HL, opts=OPT.EncodingOptions(field=field), backend=:workshop)
encRop = TO.EncodingResult(Pop, TOA.pmodule_from_fringe(HRop), idPop; H=HRop, opts=OPT.EncodingOptions(field=field), backend=:workshop)
T = TO.tor(encRop, encL; maxdeg=2, cache=sc)

(; tor_degrees=collect(TO.Advanced.degree_range(T)),
   tor_dims=[TO.Advanced.dim(T, t) for t in TO.Advanced.degree_range(T)],
   tor_summary=TO.describe(T))


(tor_degrees = [0, 1, 2], tor_dims = [1, 0, 0], tor_summary = (kind = :tor_space, model = :first, field = TamerOp.CoreModules.CoeffFields.QQField(), nvertices = 2, degree_range = 0:2, nonzero_degrees = (0,), degree_dimensions = Dict(0 => 1), total_dimension = 1))

## 6. One coordinates/representative round-trip

If a degree is nonzero, we build the first basis class from coordinates, materialize a representative, and recover the coordinates again.


In [16]:
deg = first(TO.Advanced.degree_range(E))
deg_dim = TO.Advanced.dim(E, deg)
if deg_dim > 0
    coeffs = zeros(TO.QQ, deg_dim)
    coeffs[1] = TO.QQ(1)
    z = TO.Advanced.representative(E, deg, coeffs)
    recovered = TO.Advanced.coordinates(E, deg, z)
    (; degree=deg, input_coeffs=coeffs, recovered_coeffs=recovered, exact_round_trip=(recovered == coeffs))
else
    "No nonzero Ext degree was available for the round-trip cell."
end


"No nonzero Ext degree was available for the round-trip cell."

## 7. Export one geometry summary and one rank summary

For slides or later reference, keep the exported artifacts minimal.


In [17]:
rank_M = TO.rank_invariant(encM; opts=OPT.InvariantOptions(threads=false), cache=sc)
exports = TO.save_visuals(
    outdir,
    [
        (; stem="rectangles_regions", obj=TO.encoding_map(encM), kind=:regions),
        (; stem="rectangles_rank_heatmap", obj=rank_M, kind=:rank_heatmap),
    ];
    format=:html,
    backend=:cairomakie,
)

Dict(TO.export_stem(r) => TO.export_path(r) for r in exports)


Dict{String, String} with 2 entries:
  "rectangles_rank_heatmap" => "/home/eriknovak/Documents/duke_fall_2025/tamer-…
  "rectangles_regions"      => "/home/eriknovak/Documents/duke_fall_2025/tamer-…

## Try this next

Shift the second rectangle farther away, rerun notebook cells 2–5, and compare how `hom_dim`, `ext_dims`, and `tor_dims` change.


In [18]:
FN_shifted = TO.SyntheticData.orthant_bar_flange(
    bars=[([2, 2], [4, 4])],
    field=CM.QQField(),
)

encM2, encN2 = TO.encode((FM, FN_shifted), enc_opts; cache=sc)
E2 = TO.ext(encM2, encN2; maxdeg=2, cache=sc)

(; shifted_hom_dim=TO.hom_dimension(encM2, encN2; cache=sc),
   shifted_ext_dims=[TO.Advanced.dim(E2, t) for t in TO.Advanced.degree_range(E2)])


(shifted_hom_dim = 0, shifted_ext_dims = [0, 0, 0])